# Loan

Cloud migration of on-prem SSIS package `Select_for_Loans` (+ enrichment SPs `bi.sp_settlement`, `bi.sp_Payment`, and the `DWH.act.LegalActivity`-based legal/extrajudicial loads) which populate `[Analytic].[bi].[Loan]`.

Produces **one row per loan-party relationship** (Primary Owner + Guarantor + Co-Owner), delivered to the IT platform. On-prem schema and values are preserved as closely as possible; see the **Known Differences** cell.

## Sources (cloud)
- Perimeter + party/dates/status/rates/balances: `primaryactive.exposure` (all relationship types) enriched from `derivativeactive.DExposure` (dx_ aliased) on `ExposureKey`.
- Party group + segment: `derivativeactive.DParty` on `CDI`.
- Product description: `primary.Product` on `ProductKey`.
- Portfolio (PortfolioId + SPV): `primaryactive.portfolio` on `PortfolioName`.
- Settlement: `derivativeactive.DSettlementExposure` on `ExposureKey` (bucket / pdd / remaining amount).
- Payments: `derivativeactive.DCollection` (matched cash collections, FinalReason='Matched'), aggregated per exposure, euro via `derivative.converttoeur`.
- Strategy / queue: `primaryactive.casecode` on `AccountNumber` bridge.
- Channel / assignment: `primaryactive.assignment` on `ExposureKey`.
- Legal activity + extrajudicial: `primaryactive.interaction` (IsLegalFlag = true).
- LPS: `derivativeactive.dlegalprocessstatusagreementdebtor` (account) and `dlegalprocessstatusdebtor` (primary debtor).
- Euro conversion: `derivative.converttoeur(currency, amount, run_date)`.


In [0]:
import sys
from datetime import datetime

dbutils.widgets.removeAll()
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD, empty=latest)")

sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import functions as F, Window

env = aws_utils.get_env_prefix()
src_catalog = f"`{env}-cdh`"

run_date = dbutils.widgets.get("run_date").strip()
if not run_date:
    run_date = str(
        spark.table(f"{src_catalog}.primaryactive.exposure")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

try:
    datetime.strptime(run_date, "%Y-%m-%d")
except ValueError:
    raise ValueError(f"run_date '{run_date}' must be YYYY-MM-DD")

run_date_lit = F.lit(run_date).cast("date")

TARGET_TABLE = f"{env}-cdh-delivery.it.Loan"
sql_target = ".".join(f"`{p.strip('`')}`" for p in TARGET_TABLE.split("."))

print(f"env={env}  run_date={run_date}  target={TARGET_TABLE}")

In [0]:
# ==================== 1. Perimeter: get_exposures_by_business_date (active + migrated/closed) ====================
# On-prem bi.Loan grain = one row per account. The function get_exposures_by_business_date
# returns the full exposure universe at account grain (one row per ExposureKey, Primary Owner),
# including migrated/closed exposures when include_migrated=True. This matches on-prem bi.Loan
# coverage far better than primaryactive.exposure (which is active-only and party-level).
exposure = spark.sql(
    f"SELECT * FROM {src_catalog}.derivative.get_exposures_by_business_date(DATE'{run_date}', true)"
)

# ==================== 2. DExposure enrichment (balances, rates, dates, status) ====================
dx = (
    spark.table(f"{src_catalog}.derivativeactive.DExposure")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("ExposureKey").alias("dx_ExposureKey"),
        F.col("Currency").alias("dx_Currency"),
        F.col("OriginBank").alias("dx_OriginBank"),
        F.col("AssetSegment").alias("dx_AssetSegment"),
        F.col("ProductKey").alias("dx_ProductKey"),
        F.col("SubProductDescription").alias("dx_SubProductDescription"),
        F.col("ExposureStatus").alias("dx_ExposureStatus"),
        F.col("ExposureStatusCode").alias("dx_ExposureStatusCode"),
        F.col("ObjectiveStatus").alias("dx_ObjectiveStatus"),
        F.col("ObjectiveStatusdate").alias("dx_ObjectiveStatusdate"),
        F.col("DenouncedDate").alias("dx_DenouncedDate"),
        F.col("IsActiveFlag").alias("dx_IsActiveFlag"),
        F.col("ClosedDate").alias("dx_ClosedDate"),
        F.col("InitialAssignmentAmount").alias("dx_InitialAssignmentAmount"),
        F.col("Opendate").alias("dx_Opendate"),
        F.col("Maturitydate").alias("dx_Maturitydate"),
        F.col("OriginationAmount").alias("dx_OriginationAmount"),
        F.col("ExposureTenor").alias("dx_ExposureTenor"),
        F.col("ContractSigningDate").alias("dx_ContractSigningDate"),
        F.col("InstalmentFrequency").alias("dx_InstalmentFrequency"),
        F.col("Mechanism").alias("dx_Mechanism"),
        F.col("MechanismCode").alias("dx_MechanismCode"),
        F.col("OutstandingTotalBalance").alias("dx_OutstandingTotalBalance"),
        F.col("OnBalance").alias("dx_OnBalance"),
        F.col("OffBalance").alias("dx_OffBalance"),
        F.col("GBV").alias("dx_GBV"),
        F.col("BogBalance").alias("dx_BogBalance"),
        F.col("TotalPrincipal").alias("dx_TotalPrincipal"),
        F.col("TotalInterest").alias("dx_TotalInterest"),
        F.col("TotalAccruals").alias("dx_TotalAccruals"),
        F.col("TotalExpense").alias("dx_TotalExpense"),
        F.col("DueCommission").alias("dx_DueCommission"),
        F.col("DueAmount").alias("dx_DueAmount"),
        F.col("BaseRate").alias("dx_BaseRate"),
        F.col("SpreadRate").alias("dx_SpreadRate"),
        F.col("PenaltyRate").alias("dx_PenaltyRate"),
        F.col("N128Rate").alias("dx_N128Rate"),
        F.col("InterestRateType").alias("dx_InterestRateType"),
        F.col("BucketCode").alias("dx_BucketCode"),
        F.col("DaysPastDueWithMateriality").alias("dx_DaysPastDue"),
        F.col("Defaultdate").alias("dx_Defaultdate"),
    )
    .dropDuplicates(["dx_ExposureKey"])
)

# ==================== 3. DParty (tax number, group, segment) ====================
dparty = (
    spark.table(f"{src_catalog}.derivativeactive.DParty")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("CDI").alias("pt_CDI"),
        F.col("TaxRegistrationNumber").alias("pt_TaxRegistrationNumber"),
        F.col("GroupCode").alias("pt_GroupCode"),
        F.col("GroupName").alias("pt_GroupName"),
        F.col("Segment").alias("pt_Segment"),
    )
    .dropDuplicates(["pt_CDI"])
)

# ==================== 4. Product description ====================
product = (
    spark.table(f"{src_catalog}.primary.Product")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("ProductKey").alias("pr_ProductKey"),
        F.col("ProductDescription").alias("pr_ProductDescription"),
    )
    .dropDuplicates(["pr_ProductKey"])
)

# ==================== 5. Portfolio (PortfolioId + SPV) ====================
portfolio = (
    spark.table(f"{src_catalog}.primaryactive.portfolio")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("PortfolioName").alias("pf_PortfolioName"),
        F.col("PortfolioId").alias("pf_PortfolioId"),
        F.col("spv_qualco_AB").alias("pf_SPV"),
        F.col("Prefix").alias("pf_Prefix"),
    )
    .dropDuplicates(["pf_PortfolioName"])
)

# ==================== 6. Settlement (DSettlementExposure, primary settlement per loan) ====================
settlement = (
    spark.table(f"{src_catalog}.derivativeactive.DSettlementExposure")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("ExposureKey").alias("st_ExposureKey"),
        F.col("SettlementKey").alias("st_SettlementId"),
        F.col("SettlementAmount").alias("st_SettlementAmount"),
        F.col("RemainAmount").alias("st_SettlementRemainingAmount"),
        F.col("Bucket").alias("st_SettlementBucket"),
        F.col("DaysPastDue").alias("st_SettlementPastDueDays"),
        F.col("IsActiveFlag").alias("st_IsActiveFlag"),
        F.col("isPrimarySettlementFlag").alias("st_isPrimary"),
    )
    .filter(F.col("st_isPrimary") == True)
    .dropDuplicates(["st_ExposureKey"])
)

# ==================== 7. Payments aggregated per loan ====================
# On-prem bi.Loan.TotalPaymentAmountEuro sources from matched cash COLLECTIONS
# (DWH.trn.Collections, FinalMatchedFlag=1) - NOT trn.Payments. Cloud equivalent is the
# derivative DCollection table, which is deduplicated (on-prem had duplicate collection rows
# that double-counted the amount) and already carries the full prefixed ExposureKey.
collection = (
    spark.table(f"{src_catalog}.derivativeactive.DCollection")
    .filter(
        (F.col("BusinessDate") == run_date_lit)
        & (F.col("FinalReason") == "Matched")
        & (F.col("IsPaymentFlag") == True)
    )
    .withColumn(
        "AmountEuro",
        F.expr(
            f"{src_catalog}.derivative.converttoeur(Currency, "
            f"CAST(Amount AS decimal(19,2)), CAST('{run_date}' AS date))"
        ),
    )
)
pay_agg = collection.groupBy("ExposureKey").agg(
    F.sum("AmountEuro").cast("decimal(19,2)").alias("pa_TotalPaymentAmountEuro"),
    F.sum(
        F.when(
            F.col("ActualDate") >= F.add_months(run_date_lit, -3),
            F.col("AmountEuro"),
        ).otherwise(F.lit(0))
    ).cast("decimal(19,2)").alias("pa_Last3MPaymentAmountEuro"),
    F.max("ActualDate").alias("pa_LastPaymentActualDate"),
).withColumnRenamed("ExposureKey", "pa_ExposureKey")

# ==================== 8. CaseCode (strategy / queue) ====================
# primary.casecode has no ExposureKey; bridge via AccountNumber (= CASE_CONTR_NUM).
# On-prem picks the currently-open case, one per loan.
casecode = (
    spark.table(f"{src_catalog}.primaryactive.casecode")
    .filter((F.col("BusinessDate") == run_date_lit) & (F.col("CurrentlyOpenFlag") == 1))
    .select(
        F.col("AccountNumber").alias("cc_AccountNumber"),
        F.col("StrategyStep").alias("cc_StrategyStep"),
        F.col("ParentStrategyStep").alias("cc_Strategy"),
        F.col("StrategyType").alias("cc_StrategyArea"),
        F.col("Queue").alias("cc_Queue"),
    )
    .dropDuplicates(["cc_AccountNumber"])
)

# ==================== 9. Assignment (channel / process) ====================
# Count assignments per exposure to reproduce the on-prem double-assignment exclusion.
assignment_all = (
    spark.table(f"{src_catalog}.primaryactive.assignment")
    .filter(F.col("BusinessDate") == run_date_lit)
)
assign_cnt = (
    assignment_all.groupBy("ExposureKey")
    .agg(F.countDistinct("AssignmentKey").alias("ax_assign_count"))
    .withColumnRenamed("ExposureKey", "axc_ExposureKey")
)
assignment = (
    assignment_all.select(
        F.col("ExposureKey").alias("ax_ExposureKey"),
        F.col("AssignmentProcess").alias("ax_AssignmentProcess"),
        F.col("Channel").alias("ax_Channel"),
        F.col("ExternalCompanyFullName").alias("ax_ExternalCompanyFullName"),
        F.col("Segment").alias("ax_Segment"),
    )
    .dropDuplicates(["ax_ExposureKey"])
)

# ==================== 10. Legal activity + extrajudicial (interaction, IsLegalFlag=true) ====================
# interaction is a cumulative snapshot -> use MAX(BusinessDate).
interaction_all = spark.table(f"{src_catalog}.primaryactive.interaction").filter(
    F.col("IsLegalFlag") == True
)
max_int_bd = interaction_all.agg(F.max("BusinessDate")).collect()[0][0]
interaction = interaction_all.filter(F.col("BusinessDate") == F.lit(max_int_bd))

# Latest legal activity per exposure (flag / date / name of latest).
w_leg = Window.partitionBy("ExposureKey").orderBy(F.col("InteractionDate").desc())
legal = (
    interaction
    .withColumn("_rn", F.row_number().over(w_leg))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("ExposureKey").alias("lg_ExposureKey"),
        F.col("InteractionDate").alias("lg_LegalActivityDate"),
        F.col("InteractionType").alias("lg_LegalActivityName"),
    )
)

# Extrajudicial-denunciation letters: the ΕΞΩΔΙΚ...ΚΑΤΑΓΓΕΛ interaction-type family.
# Same source system as on-prem -> same InteractionType values. MAX date per exposure.
ej = (
    interaction
    .filter(F.col("InteractionType").like("%ΕΞΩΔΙΚ%ΚΑΤΑΓΓΕΛ%"))
    .groupBy("ExposureKey")
    .agg(F.max("InteractionDate").alias("ej_ExtrajudicialLetterAccountDate"))
    .withColumnRenamed("ExposureKey", "ej_ExposureKey")
)

# ==================== 11. LPS (account + primary debtor) ====================
lps_acc = (
    spark.table(f"{src_catalog}.derivativeactive.dlegalprocessstatusagreementdebtor")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("AgreementKey").alias("la_AgreementKey"),
        F.col("CDI").alias("la_CDI"),
        F.col("OperationalLegalStatusLevel3").alias("la_AccountLPS"),
    )
    .dropDuplicates(["la_AgreementKey", "la_CDI"])
)
lps_deb = (
    spark.table(f"{src_catalog}.derivativeactive.dlegalprocessstatusdebtor")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("CDI").alias("ld_CDI"),
        F.col("OperationalLegalStatusLevel3").alias("ld_PrimaryDebtorLPS"),
    )
    .dropDuplicates(["ld_CDI"])
)

In [0]:
df = (
    exposure
    .join(dx, F.col("ExposureKey") == F.col("dx_ExposureKey"), "left")
    .join(dparty, F.col("CDI") == F.col("pt_CDI"), "left")
    .join(product, F.col("dx_ProductKey") == F.col("pr_ProductKey"), "left")
    .join(portfolio, F.col("PortfolioName") == F.col("pf_PortfolioName"), "left")
    .join(settlement, F.col("ExposureKey") == F.col("st_ExposureKey"), "left")
    .join(pay_agg, F.col("ExposureKey") == F.col("pa_ExposureKey"), "left")
    # CaseCode bridges via AccountNumber: strip the portfolio Prefix (variable length) off ExposureKey.
    .join(casecode, F.expr("substring(ExposureKey, length(pf_Prefix) + 1, 100)") == F.col("cc_AccountNumber"), "left")
    .join(assignment, F.col("ExposureKey") == F.col("ax_ExposureKey"), "left")
    .join(assign_cnt, F.col("ExposureKey") == F.col("axc_ExposureKey"), "left")
    .join(legal, F.expr("substring(ExposureKey, length(pf_Prefix) + 1, 100)") == F.col("lg_ExposureKey"), "left")
    .join(ej, F.expr("substring(ExposureKey, length(pf_Prefix) + 1, 100)") == F.col("ej_ExposureKey"), "left")
    .join(lps_acc, (F.col("AgreementKey") == F.col("la_AgreementKey")) & (F.col("CDI") == F.col("la_CDI")), "left")
    .join(lps_deb, F.col("CDI") == F.col("ld_CDI"), "left")
)


def euro(col_name):
    """Convert a dx_ balance column to EUR via the cloud UDF (EUR passes through)."""
    return F.expr(
        f"{src_catalog}.derivative.converttoeur(dx_Currency, "
        f"CAST({col_name} AS decimal(19,2)), CAST('{run_date}' AS date))"
    )


# Expense = TotalExpense + DueCommission (sanctioned in-delivery sum of existing derivative columns).
expense_expr = (F.coalesce(F.col("dx_TotalExpense"), F.lit(0)) + F.coalesce(F.col("dx_DueCommission"), F.lit(0)))

# Settlement presence: primary settlement that is active.
has_settlement = F.col("st_SettlementId").isNotNull() & (F.col("st_IsActiveFlag") == 1)

# FinalBucket / PastDueDays: settlement-adjusted (settlement value when settled, else source from DExposure).
final_bucket = F.when(has_settlement, F.col("st_SettlementBucket")).otherwise(F.col("dx_BucketCode"))
final_pdd = F.when(has_settlement, F.col("st_SettlementPastDueDays")).otherwise(F.col("dx_DaysPastDue"))

# InstalmentFrequency: on-prem is nvarchar '{n}M' for n in 1..12, else NULL. dx value is int.
instalment_freq = F.when(
    F.col("dx_InstalmentFrequency").between(1, 12),
    F.concat(F.col("dx_InstalmentFrequency").cast("string"), F.lit("M")),
).otherwise(F.lit(None))

# Channel / LegalChannel: External-LF rule using assignment external-company name;
# both nulled when the loan has more than one assignment (on-prem double-assignment guard).
is_double = F.coalesce(F.col("ax_assign_count"), F.lit(0)) > 1
channel = (
    F.when(is_double, F.lit(None))
     .when(
         (F.col("ax_Channel") == "External LF")
         & F.col("ax_AssignmentProcess").isin("Lf_Offer", "Collection", "Collection_Judicial_Escalation", "N/A", "DNC_Offer"),
         F.col("ax_ExternalCompanyFullName"),
     )
     .otherwise(F.col("ax_Channel"))
)
legal_channel = (
    F.when(is_double, F.lit(None))
     .when(
         (F.col("ax_Channel") == "External LF")
         & F.col("ax_AssignmentProcess").isin("Lf_Offer", "Collection", "N/A"),
         F.lit(None),
     )
     .otherwise(F.col("ax_ExternalCompanyFullName"))
)
assignment_process = F.when(is_double, F.lit(None)).otherwise(F.col("ax_AssignmentProcess"))
queue = F.when(is_double, F.lit(None)).otherwise(F.col("cc_Queue"))

# QualcoCurrentlyOpenFlag: 'Yes'/'No' if an open QCA case exists for the loan.
qualco_open = F.when(F.col("cc_AccountNumber").isNotNull(), F.lit("Yes")).otherwise(F.lit("No"))

# Tranche: on-prem CASE WHEN CustomerSegment LIKE '%-B%' THEN 'Tranche B' ELSE 'Tranche A'.
tranche = F.when(F.col("pt_Segment").like("%-B%"), F.lit("Tranche B")).otherwise(F.lit("Tranche A"))

result = df.select(
    F.col("PortfolioName").cast("string").alias("PortfolioName"),
    F.col("ExposureKey").cast("string").alias("AccountNumber"),
    F.col("AgreementKey").cast("string").alias("AgreementNumber"),
    F.col("CDI").cast("string").alias("PrimaryDebtorCDI"),
    F.col("pt_TaxRegistrationNumber").cast("string").alias("TaxNumber"),
    F.col("dx_Currency").cast("string").alias("Currency"),
    F.col("dx_OriginBank").cast("string").alias("OriginBank"),
    F.col("pt_GroupCode").cast("string").alias("GroupCode"),
    F.col("pt_GroupName").cast("string").alias("GroupName"),
    F.col("dx_AssetSegment").cast("string").alias("AssetSegment"),
    F.col("dx_ProductKey").cast("string").alias("ProductKey"),
    F.col("pr_ProductDescription").cast("string").alias("ProductDesc"),
    F.col("dx_SubProductDescription").cast("string").alias("SubProductDesc"),
    F.col("dx_ExposureStatus").cast("string").alias("LoanStatus"),
    F.col("dx_DenouncedDate").cast("date").alias("DenouncedDate"),
    F.col("dx_ObjectiveStatus").cast("string").alias("ObjectiveStatus"),
    F.col("dx_ObjectiveStatusdate").cast("date").alias("ObjectiveStatusDate"),
    F.col("dx_IsActiveFlag").cast("boolean").alias("IsActiveFlag"),
    F.col("dx_ClosedDate").cast("date").alias("ClosingDate"),
    F.col("dx_InitialAssignmentAmount").cast("decimal(19,2)").alias("InitialAssignmentAmount"),
    F.col("dx_OutstandingTotalBalance").cast("decimal(19,2)").alias("OutstandingTotalBalance"),
    F.col("dx_OnBalance").cast("decimal(19,2)").alias("OnBalance"),
    F.col("dx_OffBalance").cast("decimal(19,2)").alias("OffBalance"),
    F.col("dx_TotalAccruals").cast("decimal(19,2)").alias("Accruals"),
    F.col("dx_TotalPrincipal").cast("decimal(19,2)").alias("Principal"),
    F.col("dx_TotalInterest").cast("decimal(19,2)").alias("Interest"),
    expense_expr.cast("decimal(19,2)").alias("Expense"),
    F.col("dx_GBV").cast("decimal(19,2)").alias("GBV"),
    euro("dx_OutstandingTotalBalance").cast("decimal(19,2)").alias("OutstandingTotalBalanceEuro"),
    euro("dx_OnBalance").cast("decimal(19,2)").alias("OnBalanceEuro"),
    euro("dx_OffBalance").cast("decimal(19,2)").alias("OffBalanceEuro"),
    euro("dx_TotalAccruals").cast("decimal(19,2)").alias("AccrualsEuro"),
    euro("dx_GBV").cast("decimal(19,2)").alias("GBVEuro"),
    F.col("dx_Opendate").cast("date").alias("OpenDate"),
    F.col("dx_Maturitydate").cast("date").alias("MaturityDate"),
    F.col("dx_OriginationAmount").cast("decimal(19,2)").alias("OriginationAmount"),
    F.col("dx_ExposureTenor").cast("int").alias("Tenor"),
    F.col("dx_Mechanism").cast("string").alias("AccountMechanism"),
    instalment_freq.cast("string").alias("InstalmentFrequency"),
    has_settlement.cast("boolean").alias("HasSettlement"),
    F.col("st_SettlementId").cast("int").alias("SettlementCode"),
    final_bucket.cast("int").alias("FinalBucket"),
    F.col("dx_BucketCode").cast("int").alias("SourceBucket"),
    F.col("st_SettlementBucket").cast("int").alias("SettlementBucket"),
    final_pdd.cast("int").alias("PastDueDays"),
    F.col("dx_DaysPastDue").cast("int").alias("SourcePastDueDays"),
    F.col("st_SettlementPastDueDays").cast("int").alias("SettlementPastDueDays"),
    F.col("st_SettlementAmount").cast("decimal(19,2)").alias("SettlementAmount"),
    F.col("st_SettlementRemainingAmount").cast("decimal(19,2)").alias("SettlementRemainingAmount"),
    F.col("dx_BaseRate").cast("decimal(19,6)").alias("BaseRate"),
    F.col("dx_SpreadRate").cast("decimal(19,6)").alias("SpreadRate"),
    F.col("dx_PenaltyRate").cast("decimal(19,6)").alias("PenaltyRate"),
    F.col("dx_N128Rate").cast("decimal(19,6)").alias("N128Rate"),
    F.col("dx_InterestRateType").cast("string").alias("InterestRateType"),
    channel.cast("string").alias("Channel"),
    legal_channel.cast("string").alias("LegalChannel"),
    assignment_process.cast("string").alias("AssignmentProcess"),
    F.col("pt_Segment").cast("string").alias("CustomerSegment"),
    F.col("la_AccountLPS").cast("string").alias("AccountLegalProcessStatus"),
    F.col("ld_PrimaryDebtorLPS").cast("string").alias("PrimaryDebtorLegalProcessStatus"),
    F.col("pa_TotalPaymentAmountEuro").cast("decimal(19,2)").alias("TotalPaymentAmountEuro"),
    F.col("pa_Last3MPaymentAmountEuro").cast("decimal(19,2)").alias("Last3MPaymentAmountEuro"),
    F.col("pa_LastPaymentActualDate").cast("date").alias("LastPaymentActualDate"),
    F.col("pf_SPV").cast("string").alias("SPV"),
    F.col("cc_StrategyStep").cast("string").alias("StrategyStep"),
    F.col("cc_Strategy").cast("string").alias("Strategy"),
    F.col("cc_StrategyArea").cast("string").alias("StrategyArea"),
    queue.cast("string").alias("Queue"),
    F.col("dx_Defaultdate").cast("date").alias("DefaultDate"),
    qualco_open.alias("QualcoCurrentlyOpenFlag"),
    F.when(F.col("dx_ExposureStatusCode") == 2, F.col("dx_DueAmount"))
     .when(F.col("dx_ExposureStatusCode").isin(0, 1), F.lit(0))
     .otherwise(F.lit(None)).cast("decimal(19,2)").alias("BalanceArrear"),
    run_date_lit.alias("BalanceDate"),
    F.col("la_AccountLPS").cast("string").alias("LegalProcessStatusName"),
    F.col("dx_ExposureStatusCode").cast("string").alias("LoanStatusValue"),
    F.lit(None).cast("int").alias("UnifiedCustomerId"),
    F.col("ExposureKey").cast("string").alias("LoanId"),
    F.col("pf_PortfolioId").cast("int").alias("PortfolioId"),
    F.col("CDI").cast("string").alias("PartyId"),
    F.col("AgreementKey").cast("string").alias("AgreementId"),
    F.col("st_SettlementId").cast("int").alias("SettlementId"),
    run_date_lit.alias("ReferenceDate"),
    F.lit(None).cast("string").alias("DebtNumber"),
    F.col("dx_ContractSigningDate").cast("date").alias("SigningDate"),
    F.col("dx_BogBalance").cast("decimal(19,2)").alias("BoGBalance"),
    F.when(F.col("lg_LegalActivityDate").isNotNull(), F.lit("Yes")).otherwise(F.lit("No")).alias("LegalActivityFlag"),
    F.col("lg_LegalActivityDate").cast("timestamp").alias("LegalActivityDate"),
    F.col("lg_LegalActivityName").cast("string").alias("LegalActivityName"),
    F.when(F.col("ej_ExtrajudicialLetterAccountDate").isNotNull(), F.lit("Yes")).otherwise(F.lit("No")).alias("ExtrajudicialLetterAccountFlag"),
    F.col("ej_ExtrajudicialLetterAccountDate").cast("timestamp").alias("ExtrajudicialLetterAccountDate"),
    F.col("pt_Segment").cast("string").alias("Tranche_src"),
    tranche.alias("Tranche"),
    F.from_utc_timestamp(F.current_timestamp(), "Europe/Athens").alias("Insert Date Time"),
).drop("Tranche_src")

## Known differences vs on-prem (document for the IT consumer)

1. **ID-system columns (`LoanId`, `PartyId`, `AgreementId`, `UnifiedCustomerId`, `SettlementId`).** On-prem stores DWH internal numeric IDs. By decision (proposal *a*) they are populated with cloud-equivalent business keys: `LoanId`=`ExposureKey`, `PartyId`=`CDI`, `AgreementId`=`AgreementKey`, `SettlementId`=`SettlementKey`. `UnifiedCustomerId` has no cloud equivalent and is left NULL (`TaxNumber` serves the grouping purpose).

2. **Balances are derivative-sourced (no recompute).** `OutstandingTotalBalance`, `OnBalance`, `OffBalance`, `GBV`, `BoGBalance`, `Principal` (=`TotalPrincipal`), `Interest` (=`TotalInterest`), `Accruals` (=`TotalAccruals`) come directly from `DExposure`. Balance components are populated for **Profits** loans only in both on-prem and cloud (QCA/Arotron carry null balances), so parity holds.

3. **`Expense` = `TotalExpense` + `DueCommission`.** On-prem `bi.Loan.Expense` = Expense + DueExpense + DueCommission. DExposure `TotalExpense` = Expense + DueExpense only; `DueCommission` (material, ~€96.7M concentrated in EUCLID Small-Business) is added here as a sum of two existing derivative columns.

4. **`*Euro` columns via `derivative.converttoeur`.** EUR passes through unchanged; other currencies converted at `run_date`. Improvement over on-prem `convertToEuro`, which left non-EUR NULL.

5. **Payments** aggregated in-notebook from `derivativeactive.DCollection` (matched cash **collections**, `FinalReason='Matched'` AND `IsPaymentFlag=true`). On-prem `bi.Loan.TotalPaymentAmountEuro` sources from `DWH.trn.Collections` (FinalMatchedFlag=1), NOT `DWH.trn.Payments` — the field name is misleading. `TotalPaymentAmountEuro`=SUM(converttoeur(Amount)), `Last3MPaymentAmountEuro`=SUM over last 3 months from `run_date`, `LastPaymentActualDate`=MAX(ActualDate), per `ExposureKey`. **Note:** on-prem had duplicate collection records that double-counted amounts for some loans; the cloud `DCollection` is deduplicated, so cloud values are the correct (lower) amount for affected loans — a data-quality improvement over on-prem.

6. **Settlement** from `derivativeactive.DSettlementExposure` (primary, active). `SettlementBucket`←Bucket, `SettlementPastDueDays`←DaysPastDue, `SettlementRemainingAmount`←RemainAmount, `SettlementAmount`←SettlementAmount (the loan-level allocated amount, NOT ProposedAmount which is the total settlement value), `SettlementCode`/`SettlementId`←SettlementKey. `FinalBucket`/`PastDueDays` = settlement value when a settlement exists, else the source `BucketCode`/`DaysPastDueWithMateriality` from DExposure.

7. **Legal activity & extrajudicial** from `primary.interaction` (`IsLegalFlag`=true). `LegalActivity*` = latest legal interaction per loan (flag/max date/type). `ExtrajudicialLetterAccount*` = MAX date over the extrajudicial-denunciation interaction types (ΕΞΩΔΙΚ ΚΑΤΑΓΓΕΛ family).

8. **Strategy / Queue / QualcoCurrentlyOpenFlag** from `primary.casecode` (currently-open case), joined via the `AccountNumber` bridge (ExposureKey with the 3-char portfolio prefix stripped). Populated for QCA-sourced cases; matches on-prem where strategy/queue came from the QCA case process.

9. **Channel / LegalChannel / AssignmentProcess** from `primary.assignment`. The on-prem 'External LF' rule is reproduced using `ExternalCompanyFullName` (= on-prem external-company PartyName), and all three are nulled when a loan has more than one assignment (on-prem double-assignment guard).

10. **`InstalmentFrequency`** = `DExposure.InstalmentFrequency` (int) formatted as `'{n}M'` for n in 1..12 (else NULL), matching the on-prem nvarchar values.

11. **`ObjectiveStatus` / `SigningDate` / `CustomerSegment`** come from `DExposure` (ObjectiveStatus/ContractSigningDate — populated for QCA and, via the DExposure QCA cross-reference, for the 1:1-matched Profits exposures) and `DParty.Segment` (CustomerSegment). `GroupCode`/`GroupName` from `DParty` (profits-sourced).

12. **Dropped columns (empty in on-prem prod):** `WorkableFlag`, `NonWorkableReason`, `ExtrajudicialLetterPrimaryFlag`, `ExtrajudicialLetterPrimaryDate` (0 populated rows on-prem), and staging-only `GLXLegalProcessStatusName`/`GLXLPSDate` (never reach on-prem bi.Loan).

13. **`ReferenceDate` = `run_date`** and **`Insert Date Time` = `current_timestamp()`** replace the on-prem `SourceExtraction` reference date.

In [0]:
(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(sql_target)
)

print(f"Written to {TARGET_TABLE}  (ReferenceDate={run_date})")